In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

In [ ]:
# Load Data
airlines = pd.read_csv("data/airlines.csv")
airports = pd.read_csv("data/airports.csv")
flights = pd.read_csv("data/flights.csv", low_memory=False)

### Graphing Flights over Time with Delay

In [ ]:

# --- Filter to first week, remove cancelled/diverted ---
week = flights[
    (flights['MONTH'] == 1) &
    (flights['DAY'].between(1, 7)) &
    (flights['CANCELLED'] == 0) &
    (flights['DIVERTED'] == 0) &
    flights['DEPARTURE_HOUR'].notna() &
    flights['ARRIVAL_DELAY'].notna()
].copy()
# --- Delay color/width bins ---
# Blue = on time, yellow → orange → red → dark red = increasingly late
DELAY_BINS   = [-np.inf, 0, 15, 30, 60, np.inf]
DELAY_COLORS = [
    'rgba(100,149,237,0.5)',  # blue      — on time / early
    'rgba(255,220,50,0.6)',   # yellow    — 1–15 min
    'rgba(255,140,0,0.7)',    # orange    — 15–30 min
    'rgba(220,40,40,0.8)',    # red       — 30–60 min
    'rgba(139,0,0,0.9)',      # dark red  — 60+ min
]
DELAY_WIDTHS = [0.5, 1, 2, 3, 4.5]


In [ ]:
def build_edge_traces(hour_flights):
    # Always initialize all 6 traces as empty
    on_time_lats, on_time_lons = [], []
    bin_lats  = {1: [], 2: [], 3: [], 4: [], 6: []}
    bin_lons  = {1: [], 2: [], 3: [], 4: [], 6: []}

    if not hour_flights.empty:
        routes = (
            hour_flights.groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])
            .agg(COUNT=('FLIGHT_NUMBER', 'count'), AVG_DELAY=('ARRIVAL_DELAY', 'mean'))
            .reset_index()
        )

        routes = routes.merge(
            airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']],
            left_on='ORIGIN_AIRPORT', right_on='IATA_CODE'
        ).rename(columns={'LATITUDE': 'orig_lat', 'LONGITUDE': 'orig_lon'}).drop(columns='IATA_CODE')

        routes = routes.merge(
            airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']],
            left_on='DESTINATION_AIRPORT', right_on='IATA_CODE'
        ).rename(columns={'LATITUDE': 'dest_lat', 'LONGITUDE': 'dest_lon'}).drop(columns='IATA_CODE')

        routes['DELAYED_COUNT'] = (
            hour_flights[hour_flights['ARRIVAL_DELAY'] > 5]
            .groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])
            .size()
            .reindex(pd.MultiIndex.from_frame(routes[['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']]))
            .fillna(0)
            .values
        )

        on_time = routes[routes['DELAYED_COUNT'] == 0]
        delayed = routes[routes['DELAYED_COUNT'] > 0].copy()

        for _, row in on_time.iterrows():
            on_time_lats += [row['orig_lat'], row['dest_lat'], None]
            on_time_lons += [row['orig_lon'], row['dest_lon'], None]

        if not delayed.empty:
            delayed['width'] = pd.cut(
                delayed['DELAYED_COUNT'], bins=5,
                labels=[1, 2, 3, 4, 6],
                duplicates='drop'
            ).astype(float).fillna(1)

            for _, row in delayed.iterrows():
                w = row['width']
                bin_lats[w] += [row['orig_lat'], row['dest_lat'], None]
                bin_lons[w] += [row['orig_lon'], row['dest_lon'], None]


    traces = [
        go.Scattermap(lat=on_time_lats, lon=on_time_lons, mode='lines',
                      line=dict(width=0.5, color='rgba(100,149,237,0.2)'),
                      hoverinfo='none', showlegend=False),
    ]
    for w, color in zip([1, 2, 3, 4, 6], [
        'rgba(255,220,50,0.6)',
        'rgba(255,160,0,0.7)',
        'rgba(220,80,40,0.8)',
        'rgba(200,20,20,0.85)',
        'rgba(139,0,0,0.9)',
    ]):
        traces.append(go.Scattermap(
            lat=bin_lats[w], lon=bin_lons[w], mode='lines',
            line=dict(width=w, color=color),
            hoverinfo='none', showlegend=False,
        ))

    return traces  

In [ ]:
active_iata = set(week['ORIGIN_AIRPORT']) | set(week['DESTINATION_AIRPORT'])
active_airports = airports[airports['IATA_CODE'].isin(active_iata)]

node_trace = go.Scattermap(
    lat=active_airports['LATITUDE'],
    lon=active_airports['LONGITUDE'],
    mode='markers',
    marker=dict(size=5, color='white', opacity=0.8),
    text=active_airports['IATA_CODE'] + ' — ' + active_airports['CITY'],
    hoverinfo='text',
    showlegend=False,
)

In [ ]:
# --- Build all 168 frames (7 days × 24 hours) ---
frames = []
slider_steps = []

for day in range(1,8):
    for hour in range(24):
        label = f"Jan {day:02d}  {hour:02d}:00"
        slice_ = week[(week['DAY'] == day) & (week['DEPARTURE_HOUR'] == hour)]
        edge_traces = build_edge_traces(slice_)

        frames.append(go.Frame(
            data=edge_traces + [node_trace],
            name=label
        ))
        slider_steps.append(dict(
            method='animate',
            args=[[label], dict(mode='immediate', frame=dict(duration=0, redraw=True))],
            label=f"D{day} {hour:02d}h"
        ))

In [ ]:
# --- Build all 168 frames (7 days × 24 hours) ---
frames = []
slider_steps = []

for day in range(1,8):
    for hour in range(24):
        label = f"Jan {day:02d}  {hour:02d}:00"
        slice_ = week[(week['DAY'] == day) & (week['DEPARTURE_HOUR'] == hour)]
        edge_traces = build_edge_traces(slice_)

        frames.append(go.Frame(
            data=edge_traces + [node_trace],
            name=label
        ))
        slider_steps.append(dict(
            method='animate',
            args=[[label], dict(mode='immediate', frame=dict(duration=0, redraw=True))],
            label=f"D{day} {hour:02d}h"
        ))

### Exploring Delay Propagation

In [ ]:
# Sort each aircraft's flights in order
chains = flights[
    (flights['CANCELLED'] == 0) &
    flights['TAIL_NUMBER'].notna() &
    flights['ARRIVAL_DELAY'].notna() &
    flights['DEPARTURE_DELAY'].notna()
].sort_values(['TAIL_NUMBER', 'YEAR', 'MONTH', 'DAY', 'DEPARTURE_TIME']).copy()

# Shift to get previous leg info
chains['PREV_ARRIVAL_DELAY']  = chains.groupby('TAIL_NUMBER')['ARRIVAL_DELAY'].shift(1)
chains['PREV_DESTINATION']    = chains.groupby('TAIL_NUMBER')['DESTINATION_AIRPORT'].shift(1)
chains['LEG']                 = chains.groupby('TAIL_NUMBER').cumcount()

# Only keep legs where the previous destination matches current origin
# (filters out overnight resets / data gaps)
chains = chains[
    chains['PREV_DESTINATION'] == chains['ORIGIN_AIRPORT']
].copy()

In [ ]:
# How much delay was absorbed vs passed on at turnaround
chains['DELAY_DELTA'] = chains['DEPARTURE_DELAY'] - chains['PREV_ARRIVAL_DELAY']
# Negative = airport/crew absorbed delay
# Positive = delay grew at turnaround

# Was this flight's departure delay mostly explained by the previous arrival?
chains['PROPAGATED'] = (
    (chains['PREV_ARRIVAL_DELAY'] > 15) &
    (chains['DEPARTURE_DELAY'] > 0)
)

# How much of the departure delay came from the inbound aircraft?
chains['INHERITED_RATIO'] = (
    chains['DEPARTURE_DELAY'] / chains['PREV_ARRIVAL_DELAY']
).clip(0, 1)  # cap at 1 (can't inherit more than 100%)
chains['INHERITED_RATIO'] = chains['INHERITED_RATIO'].where(
    chains['PREV_ARRIVAL_DELAY'] > 0, 0
)

In [ ]:
print("=== Propagation Rate ===")
print(chains['PROPAGATED'].value_counts(normalize=True))

print("\n=== Avg delay delta at turnaround ===")
print(chains['DELAY_DELTA'].describe())

print("\n=== Worst propagating aircraft ===")
print(
    chains[chains['PROPAGATED']]
    .groupby('TAIL_NUMBER')
    .agg(
        LEGS=('FLIGHT_NUMBER', 'count'),
        AVG_INHERITED=('PREV_ARRIVAL_DELAY', 'mean'),
        AVG_DEPARTED=('DEPARTURE_DELAY', 'mean'),
        AVG_DELTA=('DELAY_DELTA', 'mean'),
    )
    .sort_values('LEGS', ascending=False)
    .head(20)
)

print("\n=== Routes most affected by propagation ===")
print(
    chains[chains['PROPAGATED']]
    .groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])
    .agg(
        PROPAGATED_FLIGHTS=('PROPAGATED', 'sum'),
        AVG_DELAY=('DEPARTURE_DELAY', 'mean'),
    )
    .sort_values('PROPAGATED_FLIGHTS', ascending=False)
    .head(20)
)